# 06 — Robustness checks

Each check re-estimates the main DiD coefficient (Post x High exposure, firm and year fixed effects,
SE clustered by firm) under one change of sample, variable definition or specification.

* **H1 outcome:** ln(carbon intensity), H1 sample
* **H2 outcomes:** ln(emissions), ln(real revenue), ln(real carbon intensity), industry sample

All results are collected in `output/tables/robustness.csv` (and `.tex`).

In [1]:
import os, sys
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, "src")

import numpy as np
import pandas as pd
import pyfixest as pf
import sample as S

TAB = Path("output/tables")
VC = {"CRV1": "ric"}
EII = ["17", "19", "20", "23", "24"]  # energy-intensive industries: paper, refined petroleum, chemicals, non-metallic minerals, basic metals

p = S.load_panel()


def prep(df):
    df = df.copy()
    df["post_high"] = df.post * df.high_exposure
    df["ln_revenue"] = np.log(df.revenue.where(df.revenue > 0))
    return df

h1 = prep(S.main_sample(p))
h2 = prep(S.industry_sample(p))
OUTCOMES = [("H1", "ln_ci", h1), ("H2", "ln_emissions", h2), ("H2", "ln_real_revenue", h2), ("H2", "ln_real_ci", h2)]


def est(df, y, treat="post_high", fe="ric + fyear"):
    d = df.dropna(subset=[y, treat])
    m = pf.feols(f"{y} ~ {treat} | {fe}", data=d, vcov=VC)
    t = m.tidy().loc[treat]
    return {"coef": t["Estimate"], "se": t["Std. Error"], "p": t["Pr(>|t|)"], "N": m._N, "firms": d.ric.nunique()}


rows = []
def run(check, transform=None, treat="post_high", fe="ric + fyear"):
    for hyp, y, df in OUTCOMES:
        d = transform(df, y) if transform else df
        try:
            r = est(d, y, treat, fe)
        except Exception as e:
            r = {"coef": np.nan, "se": np.nan, "p": np.nan, "N": 0, "firms": 0}
            print(f"{check} / {y}: {str(e)[:80]}")
        rows.append({"check": check, "hypothesis": hyp, "outcome": y, **r})

## Checks

In [2]:
# 0. Baseline
run("0 Baseline")

# 1. Reported emissions only
run("1 Reported emissions only", lambda d, y: d[d.reported])

# 2a. Winsorize outcome at 1st/99th percentile
def winsor(d, y):
    lo, hi = d[y].quantile([.01, .99]); return d.assign(**{y: d[y].clip(lo, hi)})
run("2a Winsorized 1/99%", winsor)

# 2b. Drop firm-years with year-on-year jumps larger than a factor of 3
def drop_jumps(d, y):
    s = d.sort_values(["ric", "fyear"]); jump = s.groupby("ric")[y].diff().abs() > np.log(3)
    return s[~jump]
run("2b Drop jumps > factor 3", drop_jumps)

# 3. Balanced panel 2016-2024
def balanced(d, y):
    n = d.dropna(subset=[y]).groupby("ric").fyear.nunique(); return d[d.ric.isin(n[n == 9].index)]
run("3 Balanced panel", balanced)

# 4. Drop COVID year 2020
run("4 Without 2020", lambda d, y: d[d.fyear != 2020])

# 5. Shock dated 2021 (anticipation)
run("5 Shock year 2021", lambda d, y: d.assign(post_high=(d.fyear >= 2021).astype(int) * d.high_exposure))

# 6a. Terciles: top vs. bottom third (middle third dropped)
run("6a Top vs. bottom tercile", lambda d, y: d[d.exposure_tercile != "mid"].assign(
    post_high=lambda x: x.post * (x.exposure_tercile == "high").astype(int)))

# 6b. Sector-based treatment: energy-intensive industries (NACE 17, 19, 20, 23, 24)
run("6b Treatment = EII sector", lambda d, y: d.assign(post_high=d.post * d.nace2.isin(EII).astype(int)))

# 7. Country-by-year fixed effects (absorb national energy prices and policies)
run("7 Country x year FE", fe="ric + country^fyear")

# 8. Sector-by-year fixed effects (NACE 2-digit; identification within sectors)
run("8 Sector x year FE", fe="ric + nace2^fyear")

# 10. Without energy companies: electricity/gas supply (NACE 35) and refined petroleum (NACE 19)
run("10 Without NACE 35 and 19", lambda d, y: d[~d.nace2.isin(["35", "19"])])

# 9. Market-based Scope 2 (H1 outcome only; data from 2020)
mb = h1.assign(ln_ci_mb=np.log((h1.scope12_marketbased / h1.revenue).where(lambda x: x > 0)))
mb = mb[mb.fyear >= 2020]
r = est(mb, "ln_ci_mb")
rows.append({"check": "9 Market-based Scope 2 (2020-2024)", "hypothesis": "H1", "outcome": "ln_ci_mb", **r})

rob = pd.DataFrame(rows)
rob["stars"] = np.select([rob.p < .01, rob.p < .05, rob.p < .1], ["***", "**", "*"], "")
rob.to_csv(TAB / "robustness.csv", index=False)

C:\_repos\energy-prices-decarbonization\.venv\Lib\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 23 singleton fixed effect(s) dropped from the model.
  warnings.warn(
C:\_repos\energy-prices-decarbonization\.venv\Lib\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 21 singleton fixed effect(s) dropped from the model.
  warnings.warn(
C:\_repos\energy-prices-decarbonization\.venv\Lib\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 21 singleton fixed effect(s) dropped from the model.
  warnings.warn(
C:\_repos\energy-prices-decarbonization\.venv\Lib\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 21 singleton fixed effect(s) dropped from the model.
  warnings.warn(
C:\_repos\energy-prices-decarbonization\.venv\Lib\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 128 singleton fixed effect(s) dropped from the model.
  warnings.warn(
C:\_repos\energy-pr

## H2 with continuous exposure

Same mechanism outcomes, but treatment = Post x standardised log pre-crisis energy intensity (no median split).
Estimated with year FE (baseline), after removing firm-specific linear trends 2016-2019, and with NACE 2-digit x year FE.


In [3]:
def detrend(df, y, years=range(2016, 2020), min_obs=3):
    out = df.copy(); out[f"{y}_dt"] = np.nan
    for ric, g in df.dropna(subset=[y]).groupby("ric"):
        pre = g[g.fyear.isin(years)]
        if pre.fyear.nunique() >= min_obs:
            a, b = np.polyfit(pre.fyear, pre[y], 1)
            out.loc[g.index, f"{y}_dt"] = g[y] - (a * g.fyear + b)
    return out

h2c = h2.assign(post_exp=h2.post * h2.exposure_std)
cont = []
for y in ["ln_emissions", "ln_real_revenue", "ln_real_ci"]:
    dt = detrend(h2c, y)
    for spec, d, yy, fe in [("Baseline", h2c, y, "ric + fyear"),
                            ("Firm trends", dt.dropna(subset=[f"{y}_dt"]), f"{y}_dt", "ric + fyear"),
                            ("Sector x year FE", h2c, y, "ric + nace2^fyear")]:
        cont.append({"outcome": y, "spec": spec, **est(d, yy, treat="post_exp", fe=fe)})
cont = pd.DataFrame(cont)
cont["stars"] = np.select([cont.p < .01, cont.p < .05, cont.p < .1], ["***", "**", "*"], "")
cont.to_csv(TAB / "h2_continuous.csv", index=False)
cont.round(3)


C:\_repos\energy-prices-decarbonization\.venv\Lib\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 11 singleton fixed effect(s) dropped from the model.
  warnings.warn(


C:\_repos\energy-prices-decarbonization\.venv\Lib\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 11 singleton fixed effect(s) dropped from the model.
  warnings.warn(


C:\_repos\energy-prices-decarbonization\.venv\Lib\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 11 singleton fixed effect(s) dropped from the model.
  warnings.warn(


,outcome,spec,coef,se,p,N,firms,stars
0,ln_emissions,Baseline,-0.188,0.073,0.011,1936,230,**
1,ln_emissions,Firm trends,-0.185,0.090,0.042,1734,198,**
2,ln_emissions,Sector x year FE,-0.238,0.117,0.043,1925,230,**
3,ln_real_revenue,Baseline,-0.081,0.035,0.022,1936,230,**
4,ln_real_revenue,Firm trends,0.009,0.023,0.685,1734,198,
5,ln_real_revenue,Sector x year FE,0.053,0.053,0.325,1925,230,
6,ln_real_ci,Baseline,-0.108,0.071,0.132,1936,230,
7,ln_real_ci,Firm trends,-0.194,0.094,0.041,1734,198,**
8,ln_real_ci,Sector x year FE,-0.291,0.096,0.003,1925,230,***


## Results overview (coefficient, stars: * p<0.10, ** p<0.05, *** p<0.01)

In [4]:
wide = rob.assign(cell=rob.coef.round(3).astype(str) + rob.stars + " (" + rob.se.round(3).astype(str) + ")")
wide = wide.pivot_table(index="check", columns="outcome", values="cell", aggfunc="first")[
    ["ln_ci", "ln_ci_mb", "ln_emissions", "ln_real_revenue", "ln_real_ci"]]
wide.to_latex(TAB / "robustness.tex", na_rep="")
wide.fillna("")

outcome,ln_ci,ln_ci_mb,ln_emissions,ln_real_revenue,ln_real_ci
check,,,,,
0 Baseline,-0.209*** (0.069),,-0.255*** (0.081),-0.166*** (0.04),-0.089 (0.081)
1 Reported emissions only,-0.21*** (0.069),,-0.265*** (0.082),-0.173*** (0.038),-0.091 (0.083)
10 Without NACE 35 and 19,-0.178** (0.073),,-0.259*** (0.091),-0.162*** (0.04),-0.097 (0.092)
2a Winsorized 1/99%,-0.188*** (0.068),,-0.241*** (0.08),-0.158*** (0.039),-0.071 (0.078)
2b Drop jumps > factor 3,-0.183** (0.075),,-0.252*** (0.088),-0.172*** (0.038),-0.081 (0.089)
3 Balanced panel,-0.218** (0.085),,-0.256** (0.102),-0.187*** (0.046),-0.068 (0.105)
4 Without 2020,-0.193*** (0.068),,-0.253*** (0.084),-0.172*** (0.041),-0.081 (0.083)
5 Shock year 2021,-0.157** (0.077),,-0.23** (0.092),-0.168*** (0.039),-0.062 (0.092)
6a Top vs. bottom tercile,-0.294*** (0.096),,-0.405*** (0.113),-0.242*** (0.051),-0.163 (0.116)


## Firms per check

In [5]:
rob.pivot_table(index="check", columns="outcome", values="firms", aggfunc="first").fillna("")

outcome,ln_ci,ln_ci_mb,ln_emissions,ln_real_ci,ln_real_revenue
check,,,,,
0 Baseline,387.0,,230.0,230.0,230.0
1 Reported emissions only,387.0,,230.0,230.0,230.0
10 Without NACE 35 and 19,360.0,,203.0,203.0,203.0
2a Winsorized 1/99%,387.0,,230.0,230.0,230.0
2b Drop jumps > factor 3,387.0,,230.0,230.0,230.0
3 Balanced panel,234.0,,157.0,157.0,157.0
4 Without 2020,387.0,,230.0,230.0,230.0
5 Shock year 2021,387.0,,230.0,230.0,230.0
6a Top vs. bottom tercile,258.0,,154.0,154.0,154.0
